In [1]:
import os
from pathlib import Path
import pandas as pd
import gc
import numpy as np

BASE_PATH = Path.cwd().parents[0]
GOLD_PATH = BASE_PATH / "data" / "gold"


In [2]:
df_gold = pd.read_csv(GOLD_PATH / "la_port_ml_ready_dataset.csv")

In [3]:
df_gold.head()

,BerthTime,AvgSOG,VesselDraft,ArrivalHour,ArrivalDayOfWeek,TerminalID,BerthID,DailyCapacityTEU,DraftMargin,LOAMargin,DelayMinutes,IsDelayed,BerthFeasible,CongestionLevel
0,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
1,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
2,232.71,0.051193,6.7,13,6,T1,B1,15298,9.2,120.0,21083.23,1,1,High
3,232.71,0.051193,6.7,13,6,T1,B2,18334,8.8,198.0,21083.23,1,1,High
4,232.71,0.051193,6.7,13,6,T1,B2,18334,8.8,198.0,21083.23,1,1,High


In [4]:
df_gold.shape

(233520, 14)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)


In [6]:
df_gold.columns

Index(['BerthTime', 'AvgSOG', 'VesselDraft', 'ArrivalHour', 'ArrivalDayOfWeek',
       'TerminalID', 'BerthID', 'DailyCapacityTEU', 'DraftMargin', 'LOAMargin',
       'DelayMinutes', 'IsDelayed', 'BerthFeasible', 'CongestionLevel'],
      dtype='object')

In [7]:
from sklearn.preprocessing import LabelEncoder

df_enc = df_gold.copy()

cat_cols = ["TerminalID", "BerthID"]

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col]) + 1
    label_encoders[col] = le   # save encoders for inference


In [8]:
df_enc.head()

,BerthTime,AvgSOG,VesselDraft,ArrivalHour,ArrivalDayOfWeek,TerminalID,BerthID,DailyCapacityTEU,DraftMargin,LOAMargin,DelayMinutes,IsDelayed,BerthFeasible,CongestionLevel
0,232.71,0.051193,6.7,13,6,1,1,15298,9.2,120.0,21083.23,1,1,High
1,232.71,0.051193,6.7,13,6,1,1,15298,9.2,120.0,21083.23,1,1,High
2,232.71,0.051193,6.7,13,6,1,1,15298,9.2,120.0,21083.23,1,1,High
3,232.71,0.051193,6.7,13,6,1,3,18334,8.8,198.0,21083.23,1,1,High
4,232.71,0.051193,6.7,13,6,1,3,18334,8.8,198.0,21083.23,1,1,High


In [9]:
# Target
y = df_enc["BerthFeasible"]

In [10]:
# Features (EXCLUDE target & leakage)
FEATURE_COLS = [
    "VesselDraft",
    "ArrivalHour",
    "ArrivalDayOfWeek",
    "TerminalID",
    "BerthID"
]

X = df_enc[FEATURE_COLS]
y = df_enc["BerthFeasible"]


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


In [12]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=300,
    min_samples_split=500,
    random_state=42
)

dt_model.fit(X_train, y_train)

dt_preds = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_preds))
print(confusion_matrix(y_test, dt_preds))
print(classification_report(y_test, dt_preds))


Decision Tree Accuracy: 0.8887461459403906
[[ 4119  4366]
 [ 2129 47766]]
              precision    recall  f1-score   support

           0       0.66      0.49      0.56      8485
           1       0.92      0.96      0.94     49895

    accuracy                           0.89     58380
   macro avg       0.79      0.72      0.75     58380
weighted avg       0.88      0.89      0.88     58380



In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_leaf": [200, 300, 500],
    "min_samples_split": [300, 500, 800]
}

dt = DecisionTreeClassifier(random_state=42)

grid = GridSearchCV(
    dt,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 200, 'min_samples_split': 300}


In [14]:
best_dt = grid.best_estimator_


In [15]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

dt_preds = best_dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_preds))
print(confusion_matrix(y_test, dt_preds))
print(classification_report(y_test, dt_preds))


Decision Tree Accuracy: 0.8972422062350119
[[ 5002  3483]
 [ 2516 47379]]
              precision    recall  f1-score   support

           0       0.67      0.59      0.63      8485
           1       0.93      0.95      0.94     49895

    accuracy                           0.90     58380
   macro avg       0.80      0.77      0.78     58380
weighted avg       0.89      0.90      0.89     58380



In [16]:
train_preds = best_dt.predict(X_train)

print("Train Accuracy:", accuracy_score(y_train, train_preds))
print("Test Accuracy:", accuracy_score(y_test, dt_preds))


Train Accuracy: 0.8997030946671234
Test Accuracy: 0.8972422062350119


In [17]:
from sklearn.metrics import make_scorer, recall_score

recall_0_scorer = make_scorer(
    recall_score,
    pos_label=0
)

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={
        "max_depth": [6, 8, 10],
        "min_samples_leaf": [300, 500, 800],
        "min_samples_split": [300, 500, 800]
    },
    scoring=recall_0_scorer,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)


,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'max_depth': [6, 8, ...], 'min_samples_leaf': [300, 500, ...], 'min_samples_split': [300, 500, ...]}"
,scoring,"make_scorer(r..., pos_label=0)"
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [18]:
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=300,
    min_samples_split=300,
    class_weight={0: 3, 1: 1},  # emphasize infeasible
    random_state=42
)


In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

dt_final = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=300,
    min_samples_split=300,
    class_weight={0: 3, 1: 1},
    random_state=42
)

dt_final.fit(X_train, y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,10
,min_samples_split,300
,min_samples_leaf,300
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,"{0: 3, 1: 1}"


In [20]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

dt_test_preds = dt_final.predict(X_test)

print("Decision Tree Test Accuracy:", accuracy_score(y_test, dt_test_preds))
print(confusion_matrix(y_test, dt_test_preds))
print(classification_report(y_test, dt_test_preds))


Decision Tree Test Accuracy: 0.8603973963686193
[[ 7576   909]
 [ 7241 42654]]
              precision    recall  f1-score   support

           0       0.51      0.89      0.65      8485
           1       0.98      0.85      0.91     49895

    accuracy                           0.86     58380
   macro avg       0.75      0.87      0.78     58380
weighted avg       0.91      0.86      0.87     58380



In [21]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": dt_final.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance)



            Feature  Importance
0       VesselDraft    0.905852
1       ArrivalHour    0.064575
2  ArrivalDayOfWeek    0.028636
4           BerthID    0.000706
3        TerminalID    0.000230


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_final = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=400,
    min_samples_split=600,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X_train, y_train)

rf_preds = rf_final.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))
print(confusion_matrix(y_test, rf_preds))
print(classification_report(y_test, rf_preds))



Random Forest Accuracy: 0.8260191846522782
[[ 8048   437]
 [ 9720 40175]]
              precision    recall  f1-score   support

           0       0.45      0.95      0.61      8485
           1       0.99      0.81      0.89     49895

    accuracy                           0.83     58380
   macro avg       0.72      0.88      0.75     58380
weighted avg       0.91      0.83      0.85     58380



In [23]:
XGB_FEATURES = [
    "VesselDraft",
    "ArrivalHour",
    "ArrivalDayOfWeek",
    "TerminalID",
    "BerthID"
]


In [24]:
X = df_enc[XGB_FEATURES]
y = df_enc["BerthFeasible"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)



In [25]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.02,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=300,
    reg_alpha=5,
    reg_lambda=10,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)



,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [26]:
xgb_preds = xgb_model.predict(X_test)
print("XGBOOST Accuracy:", accuracy_score(y_test, xgb_preds))
print(confusion_matrix(y_test, xgb_preds))
print(classification_report(y_test, xgb_preds))


XGBOOST Accuracy: 0.8882536827680713
[[ 3097  3691]
 [ 1528 38388]]
              precision    recall  f1-score   support

           0       0.67      0.46      0.54      6788
           1       0.91      0.96      0.94     39916

    accuracy                           0.89     46704
   macro avg       0.79      0.71      0.74     46704
weighted avg       0.88      0.89      0.88     46704



In [27]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df_enc[FEATURE_COLS]
y = df_enc["BerthFeasible"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

gbm = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gbm.fit(X_train, y_train)

gbm_preds = gbm.predict(X_test)

print("GBM Accuracy:", accuracy_score(y_test, gbm_preds))
print(confusion_matrix(y_test, gbm_preds))
print(classification_report(y_test, gbm_preds))


GBM Accuracy: 0.8898424117848578
[[ 4167  4318]
 [ 2113 47782]]
              precision    recall  f1-score   support

           0       0.66      0.49      0.56      8485
           1       0.92      0.96      0.94     49895

    accuracy                           0.89     58380
   macro avg       0.79      0.72      0.75     58380
weighted avg       0.88      0.89      0.88     58380



In [28]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df_enc[FEATURE_COLS]
y = df_enc["BerthFeasible"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0: 3, 1: 1},  # emphasize infeasible berth
    random_state=42
)

lgbm.fit(X_train, y_train)

lgbm_preds = lgbm.predict(X_test)

print("LightGBM Accuracy:", accuracy_score(y_test, lgbm_preds))
print(confusion_matrix(y_test, lgbm_preds))
print(classification_report(y_test, lgbm_preds))


[LightGBM] [Info] Number of positive: 149686, number of negative: 25454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001353 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 166
[LightGBM] [Info] Number of data points in the train set: 175140, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.662187 -> initscore=0.673055
[LightGBM] [Info] Start training from score 0.673055
LightGBM Accuracy: 0.8729016786570744
[[ 7697   788]
 [ 6632 43263]]
              precision    recall  f1-score   support

           0       0.54      0.91      0.67      8485
           1       0.98      0.87      0.92     49895

    accuracy                           0.87     58380
   macro avg       0.76      0.89      0.80     58380
weighted avg       0.92      0.87      0.89     58380



In [29]:
MODEL_PATH = BASE_PATH / "models"

MODEL_PATH.mkdir(exist_ok=True)

print("Model folder created at:", MODEL_PATH)


Model folder created at: E:\C-DAC\Major Project\AI-Based-Maritime-Port-Intelligence\models


In [30]:
import joblib

joblib.dump(xgb_model, MODEL_PATH / "xgb_berth_feasibility.pkl")
joblib.dump(rf_final, MODEL_PATH / "rf_berth_feasibility.pkl")
joblib.dump(lgbm, MODEL_PATH / "lgbm_berth_feasibility.pkl")
joblib.dump(gbm, MODEL_PATH / "gbm_berth_feasibility.pkl")

['E:\\C-DAC\\Major Project\\AI-Based-Maritime-Port-Intelligence\\models\\gbm_berth_feasibility.pkl']

In [31]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ------------------ LOAD MODELS ------------------
rf_final = joblib.load(MODEL_PATH / "rf_berth_feasibility.pkl")
xgb_model = joblib.load(MODEL_PATH / "xgb_berth_feasibility.pkl")

# ------------------ FEATURES (same order as training) ------------------
FEATURE_COLS = [
    "VesselDraft",
    "ArrivalHour",
    "ArrivalDayOfWeek",
    "TerminalID",   # starts from 1
    "BerthID"       # starts from 1
]

# ------------------ VALIDATION SCENARIOS ------------------
# Last column = TrueBerthFeasible (0 = Infeasible, 1 = Feasible)
validation_scenarios = [
    [6.5,  8,  1, 1, 1, 1],
    [7.2, 10,  2, 1, 2, 1],
    [8.0, 12,  3, 1, 3, 1],
    [8.6, 14,  4, 1, 4, 1],
    [9.1, 16,  5, 1, 5, 1],
    [9.5, 18,  5, 2, 1, 0],
    [10.0,20,  6, 2, 2, 0],
    [7.0,  5,  0, 1, 1, 1],
    [7.8,  1,  0, 1, 2, 1],
    [10.5,22,  6, 2, 3, 0],
]

# ------------------ DATAFRAME ------------------
val_df = pd.DataFrame(
    validation_scenarios,
    columns=FEATURE_COLS + ["TrueBerthFeasible"]
)

X_val = val_df[FEATURE_COLS]
y_true = val_df["TrueBerthFeasible"]

# ------------------ MODEL VALIDATION FUNCTION ------------------
def validate_model(model, model_name):
    preds = model.predict(X_val)
    probas = model.predict_proba(X_val)

    print(f"\n{'='*15} {model_name} BERTH FEASIBILITY VALIDATION {'='*15}")
    print("Accuracy:", accuracy_score(y_true, preds))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, preds))
    print("\nClassification Report:")
    print(classification_report(
        y_true, preds,
        target_names=["Infeasible", "Feasible"]
    ))

    label_map = {0: "Infeasible", 1: "Feasible"}

    for i, (true, pred, prob) in enumerate(zip(y_true, preds, probas)):
        print(f"Scenario {i+1}")
        print(f"  Actual Feasibility    : {label_map[true]}")
        print(f"  Predicted Feasibility : {label_map[pred]}")
        print(f"  Confidence (0/1)      : {prob.round(2)}")
        print("-" * 45)

# ------------------ RUN VALIDATION ------------------
validate_model(rf_final, "Random Forest")
validate_model(xgb_model, "XGBoost")



=============== Random Forest BERTH FEASIBILITY VALIDATION ===============
Accuracy: 0.7

Confusion Matrix:
[[0 3]
 [0 7]]

Classification Report:
              precision    recall  f1-score   support

  Infeasible       0.00      0.00      0.00         3
    Feasible       0.70      1.00      0.82         7

    accuracy                           0.70        10
   macro avg       0.35      0.50      0.41        10
weighted avg       0.49      0.70      0.58        10

Scenario 1
  Actual Feasibility    : Feasible
  Predicted Feasibility : Feasible
  Confidence (0/1)      : [0. 1.]
---------------------------------------------
Scenario 2
  Actual Feasibility    : Feasible
  Predicted Feasibility : Feasible
  Confidence (0/1)      : [0.01 0.99]
---------------------------------------------
Scenario 3
  Actual Feasibility    : Feasible
  Predicted Feasibility : Feasible
  Confidence (0/1)      : [0.01 0.99]
---------------------------------------------
Scenario 4
  Actual Feasibility   

C:\Users\harsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\harsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\harsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave